# <font color=firebrick>Econometrics — M2 MBFA</font>
## Session 4 / 8 (CM) — Discrete Choice Models: Probit and Logit

**Prerequisites:** OLS estimator and multivariate regression

**Format.** We build the model, the probability framework, and the estimation principle (maximum likelihood) before Python. All examples use a simulated **insurance** scenario: does a policyholder file a claim in a given year?

**Today's objectives:**
- Understand *why* a linear model is not adequate when the outcome is binary (0/1).
- Build the latent-variable model behind probit and logit, and connect it to probability.
- Understand maximum likelihood estimation: what a likelihood is, why we maximize it (and its logarithm), and why it replaces the least-squares criterion here.
- Estimate, read, and compare a probit and a logit model.

Practice exercises for this session are provided in the companion TD notebook.

---


# 1. Why not just use OLS?

Sessions 1-3 studied models where the dependent variable is continuous - a salary, a return, an amount. Many questions in finance and insurance instead have a **binary** answer:
- Does a policyholder file a claim this year (yes/no)?
- Does a borrower default on a loan (yes/no)?
- Does a customer renew their policy (yes/no)?

We will be using the first example throughout this session. We can therefore provide some background information to help you understand the issues involved in this example: a *policyholder* is a person who holds an insurance contract, and to *file a claim* means to formally ask the insurer for compensation after a loss (an accident, a theft, etc.). Here, a "Policyholder" is not a variable - it just describes *who* the observations are. Our running example below will only ever involve two variables: `age` (the explanatory variable) and `claim` (the outcome, $1$ if a claim was filed, $0$ otherwise).

In each case, the outcome $Y$ takes only two values, conventionally $1$ ("yes") and $0$ ("no"). A natural first instinct is to just run OLS anyway: 
$$
Y_i=\beta_0+\beta_1x_i+\varepsilon_i.
$$

We call it the **linear probability model (LPM)**, because $\mathbb{E}[Y_i\mid x_i]=\beta_0+\beta_1x_i$ is literally interpreted as the probability that $Y_i=1$. The idea is intuitive, but it runs into two structural problems.

**Problem 1 - predicted probabilities can fall outside $[0,1]$.** Nothing in the OLS formula prevents $\hat\beta_0+\hat\beta_1x_i$ from being negative, or larger than $1$, for some values of $x_i$ - even though it is supposed to be a probability.

Let's illustrate why this a problem with a concrete example. Suppose we fit the LPM `claim ~ age` and obtain, say, $\hat\beta_0=0.6$ and $\hat\beta_1=-0.01$:
$$
\widehat{\text{claim}}_i = \hat\beta_0+\hat\beta_1\,\text{age}_i = 0.6 - 0.01\times\text{age}_i.
$$

So we find that each extra year of age lowers the predicted claim probability by one percentage point For an $18$-year-old policyholder, this predicts $0.6-0.01\times18=0.42$ - perfectly reasonable. But for a $90$-year-old, it predicts $0.6-0.01\times90=-0.30$ - a *negative* probability, which is meaningless. Nothing in the OLS fitting procedure "knows" that a probability cannot be negative; it simply fits the best straight line, wherever that leads.

**Problem 2 - the errors cannot be homoskedastic.**

Throughout Sessions 1-3, we assumed $\text{Var}(\varepsilon_i\mid x_i)=\sigma^2$: the error's spread around its (zero) mean is the **same fixed number** $\sigma^2$ for every observation $i$, regardless of $x_i$. This assumption is called **homoskedasticity** (literally, "same spread"), and it is what lets us use a single standard-error formula for the whole sample. Its opposite - when the error's variance *does* depend on $i$ or on $x_i$ - is called **heteroskedasticity**.

*Why would a binary $Y$ break this?*

Think about the spread of outcomes for two different groups of policyholders. In a group where the claim probability is $p(x)=0.5$, outcomes are maximally unpredictable: roughly half file a claim, half don't - a lot of variation around the average. In a group where $p(x)=0.01$, almost everyone gets $Y_i=0$ and almost no one gets $Y_i=1$ - the outcomes barely vary at all, tightly clustered around the average. So the "spread" of $Y_i$ around its mean should itself depend on how close $p(x)$ is to $0$, $0.5$, or $1$ - which already suggests $\text{Var}(Y_i\mid x_i)$ cannot be a single fixed number $\sigma^2$. To make this precise, we need to recall exactly how the variance of a $0/1$ outcome relates to its mean $p$.

*A reminder: the Bernoulli random variable*

A random variable $Z$ is called **Bernoulli** with parameter $p$ if it takes only two values: $Z=1$ with probability $p$, and $Z=0$ with probability $1-p$. Its expectation is:
$$
\mathbb{E}[Z] = 1\times p + 0\times(1-p) = p.
$$
Its variance uses a shortcut specific to $0/1$ variables. Since $Z$ only ever equals $0$ or $1$, squaring it changes nothing: $Z^2=Z$ (because $0^2=0$ and $1^2=1$). So $\mathbb{E}[Z^2]=\mathbb{E}[Z]=p$, and, using the familiar formula $\text{Var}(Z)=\mathbb{E}[Z^2]-\big(\mathbb{E}[Z]\big)^2$:
$$
\boxed{\;\text{Var}(Z) = p - p^2 = p(1-p).\;}
$$
Worth remembering: **for any $0/1$ variable, the variance is entirely determined by its mean** $p$ - there is no separate "amount of noise" to specify, unlike for a normal variable where the mean and the variance are two independent pieces of information.

##### Back to the Linear Probablity Model (LPM)

Conditional on $x_i$, the outcome $Y_i$ is itself a Bernoulli variable: it equals $1$ with probability $p(x_i)=\mathbb{E}[Y_i\mid x_i]$, and $0$ otherwise. Applying the reminder above, conditional on $x_i$:
$$
\text{Var}(Y_i\mid x_i) = p(x_i)\big(1-p(x_i)\big).
$$
Since the error is defined as $\varepsilon_i=Y_i-\mathbb{E}[Y_i\mid x_i]$, it has the same variance as $Y_i$ itself:
$$
\text{Var}(\varepsilon_i\mid x_i) = \text{Var}(Y_i\mid x_i) = p(x_i)\big(1-p(x_i)\big).
$$
This depends on $x_i$ - through $p(x_i)$ - rather than being some fixed number $\sigma^2$, the same for every observation, as was assumed throughout Sessions 1-3.

**A concrete illustration.** Take two policyholders with predicted claim probabilities $p=0.1$ (a low-risk profile) and $p=0.5$ (a high-risk profile). Their error variances are
$$
\text{Var}(\varepsilon\mid p=0.1) = 0.1\times0.9=0.09, \qquad \text{Var}(\varepsilon\mid p=0.5) = 0.5\times0.5=0.25.
$$
The second is nearly three times the first - a real, sizeable difference, not a theoretical technicality. Recall from Session 2 that the usual standard errors - and therefore the t-tests - rely on the errors having **constant** variance. Here that assumption is not a simplifying approximation that might happen to fail: it fails **by construction**, as soon as $Y_i$ is binary, for *any* dataset and *any* true relationship.

Both problems point to the same fix: instead of modeling the probability as a straight line, we need a curve that (i) stays within $[0,1]$ for every $x$, and (ii) whose slope can flatten out near $0$ and near $1$. This is exactly what an S-shaped probability curve looks like - and it is exactly what probit and logit provide.

---


# 2. A latent-variable model for binary choices

##### Why create an unobserved variable at all?

Section 1 showed that modeling the probability itself as a straight line does not work: nothing keeps it inside $[0,1]$. To fix is we don't have to abandon linear modeling. Instead, we can apply the linear model to something unbounded, and only transform it into a probability in a second step.

That unbounded quantity is what we call a **latent variable**: an unobserved variable $y_i^*$ that measures the policyholder's underlying *propensity* to file a claim - how likely they are, all things considered, to have an accident and report it. "Latent" just means *hidden*: we never observe $y_i^*$ itself, only its consequence, whether or not a claim gets filed. Crucially, $y_i^*$ is modeled exactly as we modeled a continuous outcome throughout Sessions 1-3 - an ordinary linear index, with no constraint on its range:

$$

y_i^* = \beta_0+\beta_1 x_i+\varepsilon_i,

$$

where $x_i$ could be the policyholder's age, driving experience, or any other observed characteristic, and $\varepsilon_i$ captures everything else (unobserved risk factors, luck).

At this stage, nothing probabilistic has yet been assumed about $\varepsilon_i$. We simply have a linear model for an unobserved quantity.


##### The threshold: turning a ranking into a decision

$y_i^*$ already does something useful even before we do anything else with it: since it is a number, it ranks every policyholder by propensity to claim. A policyholder with $y_i^*=2$ is, in this model, more claim-prone than one with $y_i^*=-1$. To turn this ranking into the yes/no outcome we actually observe, we need a cutoff: everyone ranked above it is predicted to claim, everyone below is not.

$$

y_i = \begin{cases}1 & \text{if } y_i^*>0,\\ 0 & \text{if } y_i^*\le0.\end{cases}

$$

**Why $0$, specifically?** It looks like an arbitrary choice, but it costs nothing. The choice of $0$ is a normalization, not an additional substantive assumption. $y_i^*$ has no natural units - we never observe it, so there is no scale on which "$0$" means something special versus, say, "$5$". Suppose the *true* cutoff were some other number $c$: a policyholder claims when $y_i^*>c$, i.e. when $y_i^*-c>0$. Substituting the latent-variable model for $y_i^*$:

$$

y_i^*-c>0 \iff (\beta_0-c)+\beta_1x_i+\varepsilon_i>0.

$$

This is *exactly* the same model with threshold $0$, but with a different intercept $\beta_0-c$ instead of $\beta_0$. Since the intercept is estimated anyway, it silently absorbs whatever the "true" cutoff might have been. Fixing the threshold at $0$ is therefore a costless normalization, not a restriction - one fewer unknown to worry about, with no loss of generality.


##### From a latent index to a probability

We do not know $\varepsilon_i$, so we cannot know for sure whether a given policyholder is above or below the cutoff - but we can compute the *probability* that they are. Picture the range of values $\varepsilon_i$ could plausibly take, weighted by how likely each is (its distribution). Being above the cutoff means $\varepsilon_i$ falls in a specific region of that distribution - so $P(y_i=1\mid x_i)$ is simply *how much of that distribution's probability lies in that region* - exactly like reading off the area under a bell curve beyond some point, which you have likely already seen in a statistics course.

Formally:

$$

P(y_i=1\mid x_i) = P(y_i^*>0\mid x_i) = P(\beta_0+\beta_1x_i+\varepsilon_i>0) = P(\varepsilon_i>-\beta_0-\beta_1x_i).

$$

At this point, we need to know something about the distribution of $\varepsilon_i$. This is the key additional step: the linear model tells us where the threshold is relative to the mean of the latent variable, but the distribution of the error tells us how much probability lies on either side of that threshold.

If the distribution of $\varepsilon_i$ is **symmetric around zero** (true of both distributions we will use), then $P(\varepsilon_i>-a)=P(\varepsilon_i<a)$ for any $a$ - the area beyond $-a$ on one side equals the area below $a$ on the other - so this simplifies to:

$$

\boxed{\;P(y_i=1\mid x_i) = P(\varepsilon_i<\beta_0+\beta_1x_i) = F(\beta_0+\beta_1x_i),\;}

$$

where $F$ is the **cumulative distribution function (CDF)** of $\varepsilon_i$: by definition, $F(a)=P(\varepsilon_i\le a)$ is the probability that $\varepsilon_i$ falls below the threshold $a$ - precisely the area we just described.

**Why a CDF is exactly the right tool here.** The linear index $\beta_0+\beta_1x_i$ can be *any* real number - arbitrarily negative, arbitrarily positive, exactly as in OLS. A CDF is precisely a function that takes any real number as input and returns a number in $[0,1]$ - by definition, since it is itself a probability. Even better: as long as $\varepsilon_i$'s distribution has no "gaps" (true of the normal and logistic distributions), $F$ is **strictly increasing** - a larger index always maps to a strictly larger probability. This means $F$ sets up a perfect one-to-one correspondence (a *bijection*) between the whole real line and the interval $(0,1)$: nothing is lost by going from $\beta_0+\beta_1x_i$ to $F(\beta_0+\beta_1x_i)$ - the *ranking* of observations by index is exactly preserved as a ranking by probability, and one could always recover the index from the probability by inverting $F$. Using a CDF here is therefore not a new modeling assumption piled on top of the linear model - it is simply a change of scale, relabeling the same linear index onto the $[0,1]$ scale that a probability must live on.

##### What exactly does the error distribution do?

It is useful to distinguish two ideas that are sometimes mixed together. The linear part $\beta_0+\beta_1x_i$
determines the location of the threshold relative to the center of the latent distribution. The distribution of $\varepsilon_i$ determines how probability is spread around that center. For example, consider a policyholder for whom $\beta_0+\beta_1x_i=1$:
- If the errors are tightly concentrated around zero, most of their probability mass lies below $1$, so the probability of a claim will be relatively high.
- If the errors are more dispersed, more probability lies above $1$, so the probability of a claim will be lower.

Thus, choosing a distribution for $\varepsilon_i$ is not just a technical detail: it determines exactly how the latent index is converted into a probability.

##### What does a "complete probabilistic model" mean?

At this point we can make an important distinction with the OLS model. The equation $y_i^*=\beta_0+\beta_1x_i+\varepsilon_i$ alone specifies a relationship between the latent outcome and $x_i$, but it does not specify the probability distribution of $y_i^*$. Once we specify, for example:

$$
\varepsilon_i\sim\mathcal N(0,1).
$$

We can derive the entire conditional distribution of the latent variable:

$$
y_i^*\mid X_i=x_i
\sim
\mathcal N(\beta_0+\beta_1x_i,1).
$$

We now know not only the center of the latent distribution, but also its shape and dispersion. This is what is meant here by a probabilistic model: we have specified a complete probability distribution for the unobserved outcome conditional on the observed covariate.

##### Why does the normal density contain an exponential?

For the probit model - one of the two models we present in this session -, we assume $\varepsilon_i\sim\mathcal N(0,1)$. The standard normal density is:

$$
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\varepsilon^2}{2}\right).
$$

The exponential is simply part of the definition of the normal distribution. Its form also gives useful intuition. Because the exponent contains $-\varepsilon^2$:
- errors close to zero have relatively high density;
- positive and negative errors of the same magnitude have the same density;
- very large errors have rapidly decreasing density.

For example, the density depends on the squared distance from zero: $\varepsilon^2$. This squared distance will later reappear in the likelihood and explain the connection between maximum likelihood and least squares. More generally, if:

$$
Y_i\mid X_i=x_i
\sim
\mathcal N(\beta_0+\beta_1x_i,\sigma^2),
$$

then its density is:

$$
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left[
-\frac{(y_i-\beta_0-\beta_1x_i)^2}{2\sigma^2}
\right].
$$

Every part of this expression now has an interpretation:
- $\beta_0+\beta_1x_i$ is the conditional mean;
- $y_i-(\beta_0+\beta_1x_i)$ is the deviation from that mean;
- the square measures the size of that deviation;
- $\sigma^2$ controls the dispersion;
- the exponential is the characteristic shape of the normal density.

For the binary-choice model, we do not need to observe $y_i^*$ to use this distribution. We only need it to derive the probability of the observed binary outcome.

##### Independence and identical distribution: what does iid mean?

The exact notation is:

$$
\varepsilon_i\overset{iid}{\sim}\mathcal N(0,\sigma^2),
$$

Here, two distinct assumptions are being made. Identically distributed means that all errors follow the same probability distribution:

$$
\varepsilon_1,\ldots,\varepsilon_N
\sim
\mathcal N(0,\sigma^2).
$$

In particular, they have the same mean and variance. Independent means that knowing one individual's error gives us no information about another individual's error. This second assumption becomes especially important when we move from one observation to the whole sample. If observations are independent, their joint probability is the product of their individual probabilities:

$$
\prod_{i=1}^N P(\text{observation }i).
$$

This is why products will appear in the likelihood. Thus, the different assumptions play different roles:

<center>

| Hypothèse | Rôle |
| :--- | :--- |
| Normalité | Spécifie la forme de la distribution. |
| Distribution identique | Utilise la même distribution pour chaque observation. |
| Indépendance | Permet d'écrire la probabilité conjointe sous forme de produit. |
| Espérance nulle (Moyenne $0$) | Centre la distribution des erreurs autour de zéro. |
| Variance $\sigma^2$ | Détermine sa dispersion. |

</center>

For the probit model, the latent scale is normalized by setting the error variance to $1$. This is necessary because $y_i^*$ is never observed: its scale cannot be identified from the binary outcome alone.

##### Two choices of $F$: probit and logit, with actual numbers

We still need to pick a distribution for $\varepsilon_i$ - equivalently, a specific $F$. Two choices are standard:

- **Probit**: $\varepsilon_i$ is standard normal, so $F=\Phi$, the standard normal CDF.

- **Logit**: $\varepsilon_i$ follows a *logistic* distribution, so $F=\Lambda$, the logistic CDF, $\Lambda(z)=\dfrac{e^z}{1+e^z}$.

Both are symmetric, smooth, S-shaped curves climbing from $0$ to $1$. Concretely, for a few values of the index $z=\beta_0+\beta_1x_i$:

<center>

| $z$ | $-2$ | $-1$ | $0$ | $1$ | $2$ |
|---|---|---|---|---|---|
| $\Phi(z)$ | $0.023$ | $0.159$ | $0.500$ | $0.841$ | $0.977$ |
| $\Lambda(z)$ | $0.119$ | $0.269$ | $0.500$ | $0.731$ | $0.881$ |

</center>

Both tables show the same S-shape: small near very negative $z$, $0.5$ at $z=0$, close to $1$ for large $z$. But notice they are not identical: at $z=2$, $\Phi$ already reaches $0.977$ while $\Lambda$ only reaches $0.881$ - the logistic distribution has **more spread** (a larger variance) than the standard normal, so it takes a *larger* index to push $\Lambda$ as close to $0$ or $1$ as $\Phi$ gets.

This is exactly why, once we estimate $\beta_0,\beta_1$ separately for each model on the same data in Section 6, the **logit coefficients come out larger in absolute value than the probit ones** - they are compensating for the logistic distribution's wider spread, not describing a genuinely stronger relationship.

The plot below draws both curves over a wider range of $z$, with the five values from the table marked as dots - so you can see directly, on the same picture, what the numbers above describe.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

z = np.linspace(-6, 6, 400)
Phi_z = norm.cdf(z)                  # standard normal CDF: the probit F
Lambda_z = 1 / (1 + np.exp(-z))      # logistic CDF: the logit F

z_table = np.array([-2, -1, 0, 1, 2])   # the same 5 values shown in the table above

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(z, Phi_z, color='steelblue', linewidth=2, label=r'$\Phi(z)$ - probit')
ax.plot(z, Lambda_z, color='firebrick', linestyle='--', linewidth=2, label=r'$\Lambda(z)$ - logit')

ax.scatter(z_table, norm.cdf(z_table), color='steelblue', zorder=5)
ax.scatter(z_table, 1 / (1 + np.exp(-z_table)), color='firebrick', zorder=5)

ax.axhline(0.5, color='grey', linestyle=':', linewidth=1)
ax.axvline(0, color='grey', linestyle=':', linewidth=1)

ax.set_xlabel(r'Index $z=\beta_0+\beta_1x$')
ax.set_ylabel(r'$P(y=1\mid x) = F(z)$')
ax.set_title(r'The two choices of $F$: standard normal CDF vs. logistic CDF')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()


For now, everything below holds for *either* choice - we simply write $F$.

---


# 3. From probability to estimation: maximum likelihood

In Sessions 1-3, estimation meant *minimizing the sum of squared residuals*. That criterion no longer makes clean sense here: $y_i$ is $0$ or $1$, and $F(\beta_0+\beta_1x_i)$ is a probability, not a prediction of $y_i$ on the same scale as a continuous residual. We need a different criterion, built directly from probability - the **likelihood**. Before bringing $x_i$ and $F$ back into the picture, let's see this criterion in its simplest possible form.

For a continuous outcome, we could write:

$$
\arg\min_\beta
\sum_{i=1}^N
(y_i-x_i'\beta)^2.
$$

For a binary outcome, however, $y_i$ is either $0$ or $1$, while $F(\beta_0+\beta_1x_i)$ is a probability. The squared difference between the two is not the natural probabilistic criterion we want to use. We therefore need a criterion that is built directly from the probability model. That criterion is maximum likelihood.

**A simple example first: estimating a constant probability.** Suppose, for a moment, we ignore $x_i$ entirely and just want to estimate a single probability $p$ that a claim occurs - like asking "what is the probability of heads for a possibly unfair coin?" from a handful of flips. We observe four policyholders: $y=(1,0,1,1)$ - three claims out of four. Intuitively, your best guess for $p$ is almost certainly "$3/4$", the observed proportion. Let's see this is exactly what "maximizing the likelihood" means. For a *candidate* value of $p$, the probability of observing exactly this data (three $1$'s, one $0$), since the four policyholders are independent, is
$$
L(p) = P(y_1=1)P(y_2=0)P(y_3=1)P(y_4=1) = p\times(1-p)\times p\times p = p^3(1-p).
$$
Let's compute $L(p)=p^3(1-p)$ for a few candidate values:

<center>

| $p$ | $0.3$ | $0.5$ | $0.75$ | $0.9$ |
|---|---|---|---|---|
| $L(p)$ | $0.019$ | $0.063$ | $0.105$ | $0.073$ |

</center>

$L(p)$ is highest at $p=0.75$ - higher than at $p=0.5$, and higher than at $p=0.9$. Calculus confirms $p=0.75$ is the exact maximum (differentiate $\log L(p)=3\log p+\log(1-p)$ with respect to $p$, set it to zero, and solve - you get $p=3/4$). **Maximum likelihood simply formalizes the intuitive guess** - "the best estimate of a probability is the observed frequency" - into a general principle that, as we now see, also works when there is no such obvious shortcut.

**The likelihood of a single observation, with covariates.** Now bring $x_i$ back. The model of Section 2 says $P(y_i=1\mid x_i)=F(\beta_0+\beta_1x_i)=p_i$ - the probability of a claim is no longer a single constant $p$, but varies with $x_i$. 

Exactly as above, we write the probability of observing the actual outcome $y_i$ - whichever of the two it was - as a single formula:
$$
L_i(\beta_0,\beta_1) = F(\beta_0+\beta_1x_i)^{y_i}\,\big(1-F(\beta_0+\beta_1x_i)\big)^{1-y_i}.
$$
The exponents $y_i$ and $1-y_i$ act as an on/off switch, exactly as $p$ and $1-p$ did above: if $y_i=1$, only $F(\beta_0+\beta_1x_i)$ survives; if $y_i=0$, only $1-F(\beta_0+\beta_1x_i)$ does.

**The likelihood of the whole sample.** Assuming the observations are independent, the probability of observing the entire sample exactly as it came in is the product of the individual likelihoods - exactly the same logic as multiplying $p\times(1-p)\times p\times p$ above, now with a different probability for each $i$:
$$
L(\beta_0,\beta_1) = \prod_{i=1}^N L_i(\beta_0,\beta_1) = \prod_{i=1}^N F(\beta_0+\beta_1x_i)^{y_i}\,\big(1-F(\beta_0+\beta_1x_i)\big)^{1-y_i}.
$$

**Why we take the logarithm.** Maximizing $L(\beta_0,\beta_1)$ directly is numerically awkward: it is a product of $N$ numbers between $0$ and $1$, so it shrinks fast. Concretely, in our simulated dataset of $N=2\,000$ policyholders (Section 4 below), if a typical individual likelihood were around $0.3$, the raw product would already be of the order of $10^{-1000}$ - vastly smaller than the smallest number a computer can represent (about $10^{-308}$): it would silently be rounded down to an exact $0$, useless for any optimization. The standard fix is to maximize the **logarithm** of $L$ instead:
$$
\ell(\beta_0,\beta_1) = \log L(\beta_0,\beta_1) = \sum_{i=1}^N\Big[y_i\log F(\beta_0+\beta_1x_i) + (1-y_i)\log\big(1-F(\beta_0+\beta_1x_i)\big)\Big].
$$
This turns an unwieldy, vanishingly small product into a manageable **sum** - using $\log(ab)=\log a+\log b$ - and does not change *where* the maximum is located: the logarithm is an increasing function, so it preserves the ranking of candidate $(\beta_0,\beta_1)$'s. Whatever values maximize $L$ also maximize $\ell$. This is exactly what `statsmodels` reports as "Log-Likelihood" in its output.

**Maximum likelihood estimation.**
$$
\boxed{\;(\hat\beta_0,\hat\beta_1) = \underset{\beta_0,\beta_1}{\arg\max}\; \ell(\beta_0,\beta_1).\;}
$$
In words: among all possible values of $(\beta_0,\beta_1)$, **Maximum Likelihood Estimation (MLE)** picks the ones that make the data we actually observed as probable as possible - exactly what we did by hand, by trial, in the coin-flip example above. This is the same spirit as the least-squares program of Session 2 ($\hat\beta=\arg\min_\beta$ sum of squared residuals): only the criterion being optimized has changed, from a *distance* (sum of squared residuals) to a *probability* (the likelihood).

**Why there is no closed-form formula here.** In the coin-flip example, we could solve $d\ell/dp=0$ by hand and get the clean answer $p=k/n$. With covariates, setting $\partial\ell/\partial\beta_0=0$ and $\partial\ell/\partial\beta_1=0$ gives equations that cannot be solved by hand or with a simple formula, because $F$ (whether $\Phi$ or $\Lambda$) is a nonlinear function of $\beta$. `statsmodels` instead solves them **numerically**: it starts from a guess and iteratively improves it until $\ell(\beta_0,\beta_1)$ can no longer be increased. This is why fitting a probit or logit model takes visibly longer than OLS, and why the summary table reports "converged: True" - confirmation that this numerical search succeeded.


##### A final connection with OLS

The connection between maximum likelihood and OLS is not accidental. Suppose instead that we had a continuous outcome and the model $Y_i=X_i'\beta+\varepsilon_i,$ with $\varepsilon_i\overset{iid}{\sim}\mathcal N(0,\sigma^2)$, which gives :

$$
Y_i\mid X_i=x_i
\sim
\mathcal N(x_i'\beta,\sigma^2).
$$

So the density of one observation is:

$$
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left[
-\frac{(y_i-x_i'\beta)^2}{2\sigma^2}
\right].
$$

Because the observations are independent, the likelihood is the product of these densities. Taking logs gives:

$$
-\frac{N}{2}\log(2\pi)
-\frac{N}{2}\log(\sigma^2)
-\frac{1}{2\sigma^2}
\sum_{i=1}^N
(y_i-x_i'\beta)^2.
$$

For a given $\sigma^2$, maximizing this expression with respect to $\beta$ is exactly equivalent to minimizing:

$$
\sum_{i=1}^N
(y_i-x_i'\beta)^2.
$$

This gives a useful interpretation of the familiar least-squares criterion:

> OLS can be understood as maximum likelihood under a particular probabilistic assumption: normally distributed errors.

The binary-choice models studied here use the same general principle. The difference is that the probability model is no longer Gaussian for the observed outcome $y_i$. We use a Bernoulli probability whose parameter is generated by the latent-variable model: $F(\beta_0+\beta_1x_i)$.

The estimation principle remains the same:

$$
\boxed{
\text{Specify a probability model}
\rightarrow
\text{write the likelihood}
\rightarrow
\text{maximize it}
}
$$

Only the probability model, and therefore the likelihood criterion, has changed.

---


# 4. Estimating and comparing probit and logit

Before touching real data, we generate our own data from a *known* Data-Generating Process (DGP) - exactly as in Session 1's Monte Carlo simulation. This time we know the true $(\beta_0,\beta_1)$ of a *latent-variable* model, and we can check that maximum likelihood recovers them - and, since we now have two candidate models, we can compare them directly on the same data.

**The scenario.** A motor insurer wants to understand what drives whether a policyholder files a claim in a given year. We use a single explanatory variable, the policyholder's (standardized) age, and simulate the latent propensity to file a claim:
$$
y_i^* = \beta_0 + \beta_1\,\text{age}_i^{\text{std}} + \varepsilon_i, \qquad \varepsilon_i\sim\mathcal{N}(0,1)\text{ iid},
$$
with $\beta_0=-0.5$ and $\beta_1=-0.8$ (older, more experienced policyholders are assumed less likely to file a claim), and we observe $y_i=1\{y_i^*>0\}$.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(seed=2024)

N = 2000
beta0_true, beta1_true = -0.5, -0.8

age = rng.uniform(18, 75, size=N)                       # policyholder age, in years
age_std = (age - age.mean()) / age.std()                # standardized, for a well-behaved latent scale

eps = rng.normal(0, 1, size=N)                           # errors ~ N(0,1): the probit assumption
ystar = beta0_true + beta1_true * age_std + eps          # the latent propensity to claim
claim = (ystar > 0).astype(int)                          # what we actually observe

insurance = pd.DataFrame({'age': age, 'age_std': age_std, 'claim': claim})
print("Share of policyholders who filed a claim:", insurance['claim'].mean())
insurance.head()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(insurance['age'], insurance['claim'], alpha=0.15, color='firebrick')
ax.set_xlabel('Age')
ax.set_ylabel('Claim filed (1 = yes)')
ax.set_title('Simulated data: claims by age')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()


**Estimating both models.** We fit a probit ($F=\Phi$, matching our DGP exactly, since we simulated $\varepsilon_i\sim\mathcal{N}(0,1)$) and a logit ($F=\Lambda$) on the *exact same data*, side by side - so any difference between the two comes only from the choice of $F$, nothing else.


In [ ]:
res_probit = smf.probit('claim ~ age_std', data=insurance).fit()
print(res_probit.summary())

In [ ]:
res_logit = smf.logit('claim ~ age_std', data=insurance).fit()
print(res_logit.summary())


**Task.** Compare `res_probit.params` to the true values $\beta_0=-0.5$, $\beta_1=-0.8$: they should be close, since probit's assumption ($\varepsilon_i\sim\mathcal{N}(0,1)$) exactly matches our DGP. Now compare `res_logit.params['age_std']` to `res_probit.params['age_std']`: the logit coefficient should be noticeably larger in absolute value - roughly by a factor of $1.6$-$1.8$ - the same scale difference between $\Phi$ and $\Lambda$ you already saw numerically in Section 2. Finally, read off the z-statistic and p-value for `age_std` in each table: the logic is exactly the t-test of Session 2, except the test statistic is called a **z-statistic** here (compared to a standard normal rather than a t-distribution, because MLE estimates are only *asymptotically* normal - approximately so in large samples, rather than exactly normal in any finite sample as under the stronger OLS assumptions).

**Why coefficients are not marginal effects, for either model.** In OLS, $\hat\beta_1$ *is* the marginal effect of $x$ on $y$: $\partial \hat y/\partial x=\hat\beta_1$, a constant. Here, for *either* model, the marginal effect of $x$ on the probability $P(y=1\mid x)=F(\beta_0+\beta_1x)$ is, by the chain rule,
$$
\frac{\partial P(y=1\mid x)}{\partial x} = f(\beta_0+\beta_1x)\,\beta_1,
$$
where $f$ is the *density* associated with $F$ (the standard normal density $\phi$ for probit, the logistic density for logit) - not $F$ itself. This depends on $x$ - exactly the non-constant, flattening-out-near-the-bounds effect we wanted back in Section 1. In practice, software reports the **average marginal effect**: this expression evaluated at each observation's $x_i$, then averaged across the sample.


In [ ]:
margeff_probit = res_probit.get_margeff(at='overall')
print(margeff_probit.summary())


       Probit Marginal Effects       
Dep. Variable:                  claim
Method:                          dydx
At:                           overall
                dy/dx    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
age_std       -0.2226      0.006    -37.441      0.000      -0.234      -0.211


In [ ]:
margeff_logit = res_logit.get_margeff(at='overall')
print(margeff_logit.summary())


**Task.** Compare the average marginal effect on `age_std` for probit and for logit. Unlike the raw coefficients (a factor of about $1.7$ apart), these should be **much closer** to one another. The reason: a marginal effect is measured directly on the probability scale, the same scale for both models, whereas the raw coefficient $\hat\beta_1$ is measured on each model's own latent-index scale, which is where $\Phi$ and $\Lambda$ differ. Marginal effects, not raw coefficients, are the fair way to compare probit and logit.

**Comparing predicted probabilities.** Coefficients differ in scale, and marginal effects already look close - the most direct comparison of all is the predicted probability curve itself, for a range of ages.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

age_grid = np.linspace(insurance['age'].min(), insurance['age'].max(), 200)
age_grid_std = (age_grid - insurance['age'].mean()) / insurance['age'].std()

pred_probit = res_probit.predict(pd.DataFrame({'age_std': age_grid_std}))
pred_logit = res_logit.predict(pd.DataFrame({'age_std': age_grid_std}))

ax.plot(age_grid, pred_probit, color='steelblue', label='probit', linewidth=2)
ax.plot(age_grid, pred_logit, color='firebrick', linestyle='--', label='logit', linewidth=2)
ax.set_xlabel('Age')
ax.set_ylabel('Predicted probability of a claim')
ax.set_title('Probit vs. logit: predicted claim probability by age')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()


**Task.** Do the two curves look meaningfully different? This is the usual finding, now confirmed three times over - coefficients, marginal effects, and predicted probabilities: probit and logit are two reasonable choices of $F$ that agree closely in substance, even though their raw coefficients live on different scales.

---

## Summary

| Step       | Question | Tool |
|--------|---|---|
| Section 1 | Why not OLS on a 0/1 outcome? | Predicted probabilities can leave $[0,1]$; errors are heteroskedastic by construction |
| Section 2 | How do we link $x$ to a probability? | A latent variable $y^*=\beta_0+\beta_1x+\varepsilon$, and $P(y=1\mid x)=F(\beta_0+\beta_1x)$ |
| Section 3 | How do we estimate $\beta$? | Maximum likelihood: $\hat\beta=\arg\max_\beta \ell(\beta)$, solved numerically |
| Section 4 | Does it work, and probit or logit? | Simulated data with a known $\beta$: both recover it; they agree on marginal effects and predictions, differing only in the scale of their raw coefficients |

Practice exercises applying all of this are provided in the companion TD notebook.
